# 基于时间序列动量的策略

本节用最简单的时间序列动量思路做向量化回测：

核心规则：如果某个品种近期收益为正，就做多；为负，就做空。只和它自己的历史收益比较。

我们以美元计价的黄金价格（`XAU=`）为例，从“只看上一根收益”逐步推广到“看最近若干根收益的均值”，再观察该策略在日内数据上的表现，最后用一个标准化的回测类 `MomVectorBacktester` 加入交易成本。

## 0. 准备工作

先导入需要的库，并读入日终收盘价数据。数据来自书中提供的公开 CSV 文件，包含多个品种，我们取其中的黄金 `XAU=`。

In [1]:
import numpy as np
import pandas as pd
from pylab import mpl, plt

plt.style.use('seaborn-v0_8')
mpl.rcParams['font.family'] = 'serif'
%matplotlib inline

In [2]:
url = 'http://hilpisch.com/pyalgo_eikon_eod_data.csv'
raw = pd.read_csv(url, index_col=0, parse_dates=True).dropna()

In [3]:
raw.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 2516 entries, 2010-01-04 to 2019-12-31
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL.O  2516 non-null   float64
 1   MSFT.O  2516 non-null   float64
 2   INTC.O  2516 non-null   float64
 3   AMZN.O  2516 non-null   float64
 4   GS.N    2516 non-null   float64
 5   SPY     2516 non-null   float64
 6   .SPX    2516 non-null   float64
 7   .VIX    2516 non-null   float64
 8   EUR=    2516 non-null   float64
 9   XAU=    2516 non-null   float64
 10  GDX     2516 non-null   float64
 11  GLD     2516 non-null   float64
dtypes: float64(12)
memory usage: 255.5 KB


In [4]:
raw.head()

,AAPL.O,MSFT.O,INTC.O,AMZN.O,GS.N,SPY,.SPX,.VIX,EUR=,XAU=,GDX,GLD
Date,,,,,,,,,,,,
2010-01-04,30.572827,30.950,20.88,133.90,173.08,113.33,1132.99,20.04,1.4411,1120.00,47.71,109.80
2010-01-05,30.625684,30.960,20.87,134.69,176.14,113.63,1136.52,19.35,1.4368,1118.65,48.17,109.70
2010-01-06,30.138541,30.770,20.80,132.25,174.26,113.71,1137.14,19.16,1.4412,1138.50,49.34,111.51
2010-01-07,30.082827,30.452,20.60,130.00,177.67,114.19,1141.69,19.06,1.4318,1131.90,49.10,110.82
2010-01-08,30.282827,30.660,20.83,133.52,174.31,114.57,1144.98,18.13,1.4412,1136.10,49.84,111.37


## 1. 基础：只看上一根收益

最简单的时间序列动量策略：

- 上一根收益为正 → 做多（持仓 `+1`）
- 上一根收益为负 → 做空（持仓 `-1`）

用 `np.sign()` 取对数收益的**符号**，正好对应多/空两种仓位。

下面三步是向量化回测的固定套路：
1. 计算每日**对数收益** `returns`；
2. 由信号生成**仓位** `position`；
3. 仓位要 **`shift(1)`**（用昨天的信号决定今天的持仓，避免“偷看未来”），再乘以当天收益，得到策略收益 `strategy`。

In [ ]:
data = pd.DataFrame(raw['XAU='])
data.rename(columns={'XAU=': 'price'}, inplace=True)
data['returns'] = np.log(data['price'] / data['price'].shift(1))
data.head()

In [ ]:
# 取上一根收益的符号作为市场仓位（1 = 多头，-1 = 空头）
data['position'] = np.sign(data['returns'])
# 用昨天的仓位 * 今天的收益 = 策略收益
data['strategy'] = data['position'].shift(1) * data['returns']
# 累计收益对比：基准（买入持有） vs 策略
data[['returns', 'strategy']].dropna().cumsum().apply(np.exp).plot(figsize=(10, 6));

**结论（图 4-7）**：只看上一根收益的策略**明显跑输**直接持有黄金。单根收益噪声太大，信号几乎是随机翻转，无法捕捉趋势。

## 2. 推广：用滚动窗口的平均收益

把“只看上一根”推广为“看**最近若干根收益的均值**”，再取符号生成信号。这样能过滤掉单根的噪声。下面用最近 **3 根** 收益的均值。

In [ ]:
# 用最近 3 天收益的均值的符号作为仓位
data['position'] = np.sign(data['returns'].rolling(3).mean())
data['strategy'] = data['position'].shift(1) * data['returns']
data[['returns', 'strategy']].dropna().cumsum().apply(np.exp).plot(figsize=(10, 6));

**结论（图 4-8）**：用 3 天均值时，策略表现**明显改善**，无论绝对收益还是相对基准都更好。

但要注意：表现对窗口参数**非常敏感**。下面改成 2 天，结果就差很多（图 4-9）。

In [ ]:
# 换成最近 2 天，表现明显变差
data['position'] = np.sign(data['returns'].rolling(2).mean())
data['strategy'] = data['position'].shift(1) * data['returns']
data[['returns', 'strategy']].dropna().cumsum().apply(np.exp).plot(figsize=(10, 6));

## 3. 日内（盘中）动量

理论上，时间序列动量在**日内**可能比日间更显著。这里用 Apple（AAPL）的**1 分钟**盘中数据，同时回测窗口为 **1、3、5、7、9** 的五个动量策略。

> 注意：本节需要本地数据文件 `../data/AAPL_1min_05052020.csv`（以及可选的 `SPX_1min_05052020.csv`）。如果没有该文件，此段会读取失败，可跳过。

In [ ]:
fn = '../data/AAPL_1min_05052020.csv'
# fn = '../data/SPX_1min_05052020.csv'  # 换成标普 500 指数
data = pd.read_csv(fn, index_col=0, parse_dates=True)
data.info()

In [ ]:
# 计算盘中对数收益
data['returns'] = np.log(data['CLOSE'] / data['CLOSE'].shift(1))

# 准备要绘制的列
to_plot = ['returns']

# 对每个动量窗口分别生成仓位与策略收益
for m in [1, 3, 5, 7, 9]:
    data['position_%d' % m] = np.sign(data['returns'].rolling(m).mean())
    data['strategy_%d' % m] = (data['position_%d' % m].shift(1) * data['returns'])
    to_plot.append('strategy_%d' % m)

In [ ]:
data[to_plot].dropna().cumsum().apply(np.exp).plot(
    title='AAPL intraday 05. May 2020',
    figsize=(10, 6),
    style=['-', '--', '--', '--', '--', '--']);

**结论（图 4-10 / 4-11）**：在这段盘中数据里，五个动量策略基本都**跑赢**了标的本身（虽然有的只略微领先）。对标普 500 指数（把 `fn` 换成 `SPX_1min_05052020.csv`）也类似——五种参数配置都跑赢指数、都为正收益（**未计交易成本**）。

## 4. 标准化回测类 `MomVectorBacktester`

为了更规范地回测动量策略，书中提供了 `MomVectorBacktester` 类。相比 `SMAVectorBacktester`，它多了两个贴近真实市场的设定：

- `amount`：**初始投资金额**；
- `tc`：每笔交易的**比例交易成本**。

类的主要属性：

| 属性 | 含义 |
| --- | --- |
| `symbol` | 使用的品种（RIC 代码） |
| `start` | 数据起始日期 |
| `end` | 数据结束日期 |
| `amount` | 初始投资金额 |
| `tc` | 每笔交易的比例交易成本 |

> 交易成本对动量策略尤其重要，因为这类策略往往会频繁换仓、产生大量交易。

### 4.1 无交易成本

先复现前面的结果，但这次以 **10,000 美元** 初始资金、**3 天**动量、**零**交易成本运行。`run_strategy` 返回 `(策略最终价值, 相对基准的超额)`。

In [ ]:
import MomVectorBacktester as Mom

# 初始资金 10000，交易成本为 0
mombt = Mom.MomVectorBacktester('XAU=', '2010-1-1', '2019-12-31', 10000, 0.0)
mombt.run_strategy(momentum=3)

In [ ]:
mombt.plot_results()

**结论（图 4-12）**：3 天动量策略**跑赢**了被动持有黄金的基准。

### 4.2 加入 0.1% 交易成本

现在假设每笔交易有 **0.1%** 的比例成本，其它不变。

In [ ]:
# 每笔交易 0.1% 成本
mombt = Mom.MomVectorBacktester('XAU=', '2010-1-1', '2019-12-31', 10000, 0.001)
mombt.run_strategy(momentum=3)

In [ ]:
mombt.plot_results()

**结论（图 4-13）**：即使是很小的交易成本，也会**严重侵蚀**该策略的表现，几乎把全部超额收益吃掉。根本原因在于动量策略**换仓频繁、交易次数多**。

---

### 小结

- 时间序列动量只和品种**自身历史**比较：过去涨就做多、过去跌就做空。
- 只看单根收益噪声太大、效果差；用**滚动窗口均值**能改善，但对窗口长度**很敏感**。
- 日内动量往往比日间更明显。
- 实盘中**交易成本是关键**：动量策略交易频繁，小成本就可能抹掉全部超额收益。